## Name: Asit Jain
## Roll No: M25DE1049
## Assignment 3 - MLBD

# Part 6: Explainability in Recommender Systems
## Task 12: Model-Agnostic Explainability (For Deep Learning Models)

Use LIME (Local Interpretable Model-agnostic Explanations) to break down neural network decisions.

LIME perturbs the input, observes how predictions change, and fits a simple interpretable model locally to explain each prediction.

**Dataset:** MovieLens ml-latest-small

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'pandas', 'numpy', 'scikit-learn', 'torch', 'lime', 'matplotlib', '-q'])

import os, zipfile, urllib.request, re, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
import lime
import lime.lime_tabular
warnings.filterwarnings('ignore')
print(f'PyTorch {torch.__version__}, LIME loaded - All imports successful!')

### Step 1: Load Data and Build Features

In [ ]:
url = 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip'
zip_path = 'ml-latest-small.zip'
extract_dir = 'ml-latest-small'
if not os.path.exists(extract_dir):
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z: z.extractall('.')

movies = pd.read_csv(os.path.join(extract_dir, 'movies.csv'))
ratings = pd.read_csv(os.path.join(extract_dir, 'ratings.csv'))

# Extract year
def extract_year(title):
    match = re.search(r'\((\d{4})\)', str(title))
    return int(match.group(1)) if match else np.nan
movies['year'] = movies['title'].apply(extract_year).fillna(1995)

# One-hot genres
all_genres = sorted(set(g for gs in movies['genres'] for g in gs.split('|') if g != '(no genres listed)'))
for genre in all_genres:
    movies[f'g_{genre}'] = movies['genres'].apply(lambda x, g=genre: 1.0 if g in x.split('|') else 0.0)
genre_cols = [f'g_{g}' for g in all_genres]

# Movie features
movie_avg = ratings.groupby('movieId')['rating'].mean()
movies['avg_rating'] = movies['movieId'].map(movie_avg).fillna(ratings['rating'].mean())
movie_feat = movies.set_index('movieId')[genre_cols + ['year', 'avg_rating']]

# User genre preferences
ratings_g = ratings.merge(movies[['movieId'] + genre_cols], on='movieId')
user_prefs = {}
for uid, group in ratings_g.groupby('userId'):
    prefs = []
    for g in genre_cols:
        mask = group[g] == 1.0
        prefs.append(group.loc[mask, 'rating'].mean() if mask.sum() > 0 else 0.0)
    user_prefs[uid] = prefs
user_feat = pd.DataFrame.from_dict(user_prefs, orient='index', columns=genre_cols)

print(f'Movie features: {movie_feat.shape}, User features: {user_feat.shape}')

### Step 2: Build Combined Feature Matrix

In [ ]:
# Build combined feature vectors for (user, movie) pairs
feature_names = [f'movie_{c}' for c in genre_cols + ['year', 'avg_rating']] + [f'user_{c}' for c in genre_cols]
feature_rows = []
labels = []

for _, row in ratings.iterrows():
    uid, mid = int(row['userId']), int(row['movieId'])
    if mid not in movie_feat.index or uid not in user_feat.index:
        continue
    mf = movie_feat.loc[mid].values
    uf = user_feat.loc[uid].values
    feature_rows.append(np.concatenate([mf, uf]))
    labels.append(row['rating'])

X = np.array(feature_rows)
y = np.array(labels)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print(f'Train: {X_train_s.shape}, Test: {X_test_s.shape}, Features: {len(feature_names)}')

### Step 3: Train a Neural Network

In [ ]:
class RecNet(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1))

    def forward(self, x):
        return self.net(x).squeeze(1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RecNet(len(feature_names)).to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# Training
train_t = torch.FloatTensor(X_train_s).to(device)
train_y = torch.FloatTensor(y_train).to(device)
test_t = torch.FloatTensor(X_test_s).to(device)

batch_size = 512
best_val = float('inf')
for epoch in range(30):
    model.train()
    perm = torch.randperm(len(train_t))
    epoch_loss = 0
    for i in range(0, len(train_t), batch_size):
        idx = perm[i:i+batch_size]
        opt.zero_grad()
        loss = criterion(model(train_t[idx]), train_y[idx])
        loss.backward(); opt.step()
        epoch_loss += loss.item() * len(idx)

    model.eval()
    with torch.no_grad():
        val_pred = model(test_t).cpu().numpy()
    val_rmse = np.sqrt(mean_squared_error(y_test, val_pred))
    if val_rmse < best_val:
        best_val = val_rmse
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1}: Val RMSE = {val_rmse:.4f}')

model.load_state_dict(best_state)
model = model.to(device)
print(f'Best Val RMSE: {best_val:.4f}')

### Step 4: Create LIME Explainer

In [ ]:
# Define prediction function for LIME (takes raw numpy, returns predictions)
def nn_predict(X_raw):
    """Prediction function wrapper for LIME."""
    model.eval()
    X_scaled = scaler.transform(X_raw)
    with torch.no_grad():
        preds = model(torch.FloatTensor(X_scaled).to(device)).cpu().numpy()
    return preds

# Create LIME explainer
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train,
    feature_names=feature_names,
    mode='regression',
    random_state=42)

print('LIME explainer ready.')

### Step 5: LIME Explanations for Individual Predictions

In [ ]:
def explain_with_lime(instance_idx, num_features=10):
    """Generate and display LIME explanation for a single prediction."""
    instance = X_test[instance_idx]
    pred = nn_predict(instance.reshape(1, -1))[0]
    actual = y_test[instance_idx]

    # Generate LIME explanation
    exp = explainer.explain_instance(instance, nn_predict, num_features=num_features)

    print(f'Predicted: {pred:.2f}, Actual: {actual:.1f}')
    print(f'\nTop {num_features} feature contributions (LIME):')
    for feat, weight in exp.as_list():
        direction = '↑' if weight > 0 else '↓'
        print(f'  {direction} {feat}: {weight:+.4f}')

    # Natural language explanation
    pos_feats = [(f, w) for f, w in exp.as_list() if w > 0][:3]
    neg_feats = [(f, w) for f, w in exp.as_list() if w < 0][:2]
    if pos_feats:
        reasons = [f.split(' ')[0].replace('movie_g_', '').replace('user_g_', 'your preference for ') for f, _ in pos_feats]
        print(f'\n💡 Rating pushed UP by: {", ".join(reasons)}')
    if neg_feats:
        reasons = [f.split(' ')[0].replace('movie_g_', '').replace('user_g_', 'your preference for ') for f, _ in neg_feats]
        print(f'   Rating pushed DOWN by: {", ".join(reasons)}')

    return exp

# Explain 3 predictions
for idx in [0, 25, 100]:
    print('='*65)
    print(f'LIME Explanation for Test Sample #{idx}')
    print('='*65)
    exp = explain_with_lime(idx, num_features=8)
    print()

### Step 6: Visualize LIME Explanations

In [ ]:
# LIME built-in visualization
exp = explainer.explain_instance(X_test[0], nn_predict, num_features=10)
fig = exp.as_pyplot_figure()
fig.set_size_inches(10, 5)
plt.title('LIME Explanation: Feature Contributions to Predicted Rating')
plt.tight_layout()
plt.show()

### Step 7: LIME Stability Analysis

LIME uses random perturbations, so explanations can vary between runs. Let's measure stability.

In [ ]:
# Run LIME multiple times on the same instance and check consistency
instance = X_test[0]
n_runs = 5
feature_ranks = {}

for run in range(n_runs):
    exp = explainer.explain_instance(instance, nn_predict, num_features=10)
    for rank, (feat, weight) in enumerate(exp.as_list()):
        feat_name = feat.split(' ')[0]  # simplify feature name
        if feat_name not in feature_ranks:
            feature_ranks[feat_name] = []
        feature_ranks[feat_name].append(rank)

print('LIME Stability Analysis (5 runs on same instance):')
print(f'{"Feature":35s} | {"Avg Rank":>8s} | {"Std":>6s} | {"Appearances":>11s}')
print('-' * 70)
for feat in sorted(feature_ranks.keys(), key=lambda x: np.mean(feature_ranks[x])):
    ranks = feature_ranks[feat]
    print(f'{feat:35s} | {np.mean(ranks):>8.1f} | {np.std(ranks):>6.2f} | {len(ranks):>11d}/{n_runs}')

### Step 8: Compare LIME Explanations Across Different Rating Levels

In [ ]:
# Compare explanations for high vs low predicted ratings
preds = nn_predict(X_test)
high_idx = np.argsort(preds)[-5:]  # top 5 highest predictions
low_idx = np.argsort(preds)[:5]    # top 5 lowest predictions

def aggregate_lime_features(indices, label):
    """Aggregate LIME feature importance across multiple instances."""
    feat_weights = {}
    for idx in indices:
        exp = explainer.explain_instance(X_test[idx], nn_predict, num_features=10)
        for feat, weight in exp.as_list():
            name = feat.split(' ')[0]
            if name not in feat_weights:
                feat_weights[name] = []
            feat_weights[name].append(weight)
    print(f'\n{label} (avg predicted: {np.mean(preds[indices]):.2f}):')
    print(f'{"Feature":30s} | {"Avg Weight":>10s} | {"Direction":>9s}')
    print('-' * 55)
    sorted_feats = sorted(feat_weights.items(), key=lambda x: abs(np.mean(x[1])), reverse=True)
    for feat, weights in sorted_feats[:8]:
        avg_w = np.mean(weights)
        d = '↑ Positive' if avg_w > 0 else '↓ Negative'
        print(f'{feat:30s} | {avg_w:>+10.4f} | {d:>9s}')

aggregate_lime_features(high_idx, 'HIGH-RATED Predictions')
aggregate_lime_features(low_idx, 'LOW-RATED Predictions')

### Analysis

**Key Findings:**

1. **LIME is model-agnostic**: It works on any model (neural networks, ensembles, etc.) by treating it as a black box. This makes it ideal for explaining deep learning recommenders where internal weights are hard to interpret.

2. **Local fidelity**: LIME explains individual predictions by fitting a simple linear model in the neighborhood of each instance. This means explanations are accurate locally even if the global model is highly non-linear.

3. **Stability tradeoff**: Because LIME uses random perturbations, explanations can vary between runs. Top features are usually consistent, but lower-ranked features may shift. Running multiple times and averaging improves reliability.

4. **High vs Low predictions**: LIME reveals different feature patterns for high and low predictions. High-rated predictions often show strong user preference alignment, while low-rated ones show genre mismatches or low movie popularity.

5. **Practical use**: LIME explanations can be surfaced to users as *"This was recommended because..."* statements, improving transparency without exposing model internals.